# Advanced Tutorial Problems with Solutions  
## `yield from`: Delegation, Two-Way Communication, Exceptions, Cleanup, and Return Values

This notebook is a **new problem set** about Python generator delegation. It is written in a tutorial style:

- every major problem starts with a concrete scenario;
- the problem is divided into small logical steps;
- explanation cells appear before and after important code;
- prediction checkpoints encourage reasoning before execution;
- complete solutions and verification tests are included;
- the examples deliberately exercise more than ordinary iteration.

The central idea is:

```python
result = yield from subgenerator
```

This statement does much more than forward values. It establishes a temporary, two-way communication channel between the caller and the subgenerator.

During delegation:

- `next(...)` reaches the subgenerator;
- `send(value)` reaches the subgenerator;
- `throw(...)` reaches the subgenerator;
- `close()` is propagated to the subgenerator;
- values yielded by the subgenerator reach the caller;
- the subgenerator's return value becomes the value of the `yield from` expression.

## Learning objectives

By the end of the notebook, you should be able to:

1. explain why `yield from` is not merely a shorter `for` loop;
2. capture a subgenerator's return value;
3. design generator protocols driven by `send`;
4. inject and recover from exceptions with `throw`;
5. reason about cleanup when `close()` propagates through nested delegators;
6. compose recursive generators while aggregating return values;
7. use subgenerators as states in an interactive workflow;
8. reimplement the essential delegation protocol manually;
9. diagnose protocol errors and generator lifecycle mistakes;
10. choose when transparent delegation is appropriate—and when it is not.

## How to use this notebook

For each problem:

1. Read the scenario and requirements.
2. Pause at the **Prediction checkpoint**.
3. Try writing the requested function before opening the solution mentally.
4. Run the solution.
5. Run the verification cell.
6. Read the discussion of edge cases and design choices.

All examples target modern Python 3.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from inspect import getgeneratorstate, getgeneratorlocals
from typing import Any, Iterable, Iterator
import math

## Small testing utilities

Ordinary iteration discards the value attached to `StopIteration`. Because several problems use generator return values, we will define a helper that captures both:

- every yielded item;
- the final returned value.

In [2]:
def collect_yields_and_return(generator):
    """Exhaust a generator and return (yielded_values, return_value)."""
    yielded = []

    while True:
        try:
            yielded.append(next(generator))
        except StopIteration as stop:
            return yielded, stop.value


def capture_stop_value(callable_):
    """Execute a zero-argument callable and capture StopIteration.value."""
    try:
        callable_()
    except StopIteration as stop:
        return stop.value
    raise AssertionError("The callable did not raise StopIteration")

In [3]:
# A quick self-test for the helper.

def tiny_generator():
    yield 10
    yield 20
    return 99


items, result = collect_yields_and_return(tiny_generator())

assert items == [10, 20]
assert result == 99

items, result

([10, 20], 99)

# Problem 1 — Prove that `yield from` is not just a shorter loop

## Scenario

You are building a stateful calculator implemented as a generator.

The calculator:

- initially yields the current total;
- accepts commands with `send`;
- supports addition and multiplication;
- returns the final total when it receives `("stop", None)`.

A colleague writes a delegator using a normal `for` loop. Another writes a delegator using `yield from`.

Both delegators appear equivalent when the caller only uses `next`.

Your task is to show precisely where their behavior diverges.

## Step 1 — Define the subgenerator protocol

A command is a two-item tuple:

```python
(operation, operand)
```

Supported operations:

- `("add", number)`
- `("mul", number)`
- `("stop", None)`

The generator yields the current total after every successful command.

In [4]:
def calculator_worker(initial=0):
    total = initial

    while True:
        command = yield total

        # next(generator) is equivalent to generator.send(None).
        if command is None:
            continue

        operation, operand = command

        if operation == "add":
            total += operand
        elif operation == "mul":
            total *= operand
        elif operation == "stop":
            return total
        else:
            raise ValueError(f"Unknown operation: {operation!r}")

## Step 2 — Write two delegators

The first one forwards yielded values with a loop.

The second one delegates with `yield from`.

At first glance, these seem interchangeable.

In [5]:
def loop_delegator(initial=0):
    for value in calculator_worker(initial):
        yield value


def transparent_delegator(initial=0):
    result = yield from calculator_worker(initial)
    return result

## Prediction checkpoint

Without running the next cell, predict the results.

### Case A

```python
g = loop_delegator(10)
next(g)
g.send(("add", 5))
```

Will the second expression produce `15`?

### Case B

```python
g = transparent_delegator(10)
next(g)
g.send(("add", 5))
```

Will the second expression produce `15`?

Think about **which suspended `yield` expression receives the value passed to `send`**.

In [6]:
manual = loop_delegator(10)
delegated = transparent_delegator(10)

manual_first = next(manual)
manual_after_send = manual.send(("add", 5))

delegated_first = next(delegated)
delegated_after_send = delegated.send(("add", 5))

manual.close()
delegated.close()

manual_first, manual_after_send, delegated_first, delegated_after_send

(10, 10, 10, 15)

## Explanation

The normal loop forwards only values produced by iteration.

In the loop-based delegator, this line is the active suspension point:

```python
yield value
```

Therefore, `manual.send(("add", 5))` sends the tuple into the **delegator's** `yield`, not into the calculator's `yield`.

The loop then asks the calculator for its next item. That request is equivalent to:

```python
calculator.send(None)
```

So the calculator never receives `("add", 5)`.

By contrast, `yield from` forwards `send` directly to the subgenerator.

## Step 3 — Verify complete two-way communication

Now interact with the transparent delegator until the worker returns.

In [7]:
calc = transparent_delegator(2)

assert next(calc) == 2
assert calc.send(("add", 3)) == 5
assert calc.send(("mul", 4)) == 20
assert calc.send(("add", -8)) == 12

final_total = capture_stop_value(lambda: calc.send(("stop", None)))

assert final_total == 12
assert getgeneratorstate(calc) == "GEN_CLOSED"

final_total

12

## Design lesson

Use a normal `for` loop when you only need one-way iteration.

Use `yield from` when the delegated object participates in the generator protocol and callers may use:

- `send`;
- `throw`;
- `close`;
- a returned result.

This distinction becomes critical in coroutine-like generator designs.

# Problem 2 — Return structured results through `StopIteration.value`

## Scenario

A token parser yields every valid integer it finds.

However, the caller also needs a final report containing:

- the number of accepted tokens;
- the rejected tokens;
- the sum of accepted values.

Yielding the final report as an ordinary data item would mix two different concepts:

1. streamed results;
2. completion metadata.

Instead, the parser will **return** the report.

## Step 1 — Define a result type

A dataclass makes the completion result explicit and easy to test.

In [8]:
@dataclass(frozen=True)
class ParseReport:
    accepted_count: int
    rejected_tokens: tuple[str, ...]
    accepted_sum: int

## Step 2 — Implement the parsing subgenerator

For every valid integer token:

- convert it to `int`;
- update the report counters;
- yield the integer.

At the end, return a `ParseReport`.

In [9]:
def parse_integer_tokens(tokens):
    accepted_count = 0
    rejected = []
    accepted_sum = 0

    for token in tokens:
        try:
            value = int(token)
        except (TypeError, ValueError):
            rejected.append(str(token))
            continue

        accepted_count += 1
        accepted_sum += value
        yield value

    return ParseReport(
        accepted_count=accepted_count,
        rejected_tokens=tuple(rejected),
        accepted_sum=accepted_sum,
    )

## Step 3 — Build a reporting delegator

The delegator will:

1. stream the valid integers unchanged;
2. capture the parser's returned report;
3. yield one human-readable summary line;
4. return the same structured report to its own caller.

This demonstrates that a return value can travel through multiple generator layers.

In [10]:
def parsing_session(tokens):
    report = yield from parse_integer_tokens(tokens)

    yield (
        f"accepted={report.accepted_count}; "
        f"rejected={len(report.rejected_tokens)}; "
        f"sum={report.accepted_sum}"
    )

    return report

## Prediction checkpoint

For this input:

```python
["10", "bad", "-3", "7.5", "8"]
```

Predict:

- the values yielded by the parser;
- the summary string yielded by the delegator;
- the final `ParseReport`.

In [11]:
tokens = ["10", "bad", "-3", "7.5", "8"]

yielded, returned_report = collect_yields_and_return(parsing_session(tokens))

yielded, returned_report

([10, -3, 8, 'accepted=3; rejected=2; sum=15'],
 ParseReport(accepted_count=3, rejected_tokens=('bad', '7.5'), accepted_sum=15))

In [12]:
assert yielded == [
    10,
    -3,
    8,
    "accepted=3; rejected=2; sum=15",
]

assert returned_report == ParseReport(
    accepted_count=3,
    rejected_tokens=("bad", "7.5"),
    accepted_sum=15,
)

## Why not yield the report directly from the parser?

A streamed integer and a completion report have different meanings.

Returning the report keeps the protocol clean:

- yielded values represent data records;
- the return value represents completion status.

This pattern is useful for:

- parsers;
- ETL stages;
- file processors;
- tree traversals;
- batch jobs;
- validation pipelines.

# Problem 3 — Build a dynamically configurable analytics coroutine

## Scenario

Create a generator that maintains a moving window of numeric observations.

The caller communicates through `send`.

Supported messages:

- a number: append an observation;
- `("window", size)`: change the maximum window size;
- `"reset"`: clear all observations;
- `"stop"`: finish and return a final report.

After every accepted message, the generator yields a snapshot.

## Step 1 — Define the snapshot and completion report

A snapshot describes the current live state.

A completion report summarizes the entire session.

In [13]:
@dataclass(frozen=True)
class WindowSnapshot:
    values: tuple[float, ...]
    window_size: int
    mean: float | None


@dataclass(frozen=True)
class AnalyticsReport:
    messages_processed: int
    observations_seen: int
    final_snapshot: WindowSnapshot

## Step 2 — Write a snapshot helper

Keeping calculation logic outside the generator makes the coroutine easier to read.

In [14]:
def make_window_snapshot(values, window_size):
    mean = None if not values else sum(values) / len(values)

    return WindowSnapshot(
        values=tuple(values),
        window_size=window_size,
        mean=mean,
    )

## Step 3 — Implement the analytics engine

Important lifecycle detail:

The first `next(engine)` call advances execution to the first `yield`. It does not send an observation.

After priming, every subsequent `send(...)` provides the value of the suspended `yield` expression.

In [15]:
def moving_window_engine(initial_window=3):
    if initial_window <= 0:
        raise ValueError("initial_window must be positive")

    values = []
    messages_processed = 0
    observations_seen = 0

    message = yield make_window_snapshot(values, initial_window)
    window_size = initial_window

    while True:
        messages_processed += 1

        if message == "stop":
            final_snapshot = make_window_snapshot(values, window_size)
            return AnalyticsReport(
                messages_processed=messages_processed,
                observations_seen=observations_seen,
                final_snapshot=final_snapshot,
            )

        if message == "reset":
            values.clear()

        elif (
            isinstance(message, tuple)
            and len(message) == 2
            and message[0] == "window"
        ):
            new_size = message[1]

            if not isinstance(new_size, int) or new_size <= 0:
                raise ValueError("window size must be a positive integer")

            window_size = new_size
            del values[:-window_size]

        elif isinstance(message, (int, float)) and not isinstance(message, bool):
            observations_seen += 1
            values.append(float(message))
            del values[:-window_size]

        else:
            raise TypeError(f"Unsupported message: {message!r}")

        message = yield make_window_snapshot(values, window_size)

## Step 4 — Add a transparent service layer

The service layer delegates the entire interactive protocol.

After the engine returns, the service returns the report unchanged.

In [16]:
def analytics_service(initial_window=3):
    report = yield from moving_window_engine(initial_window)
    return report

## Guided interaction

Notice how the active window changes after each command.

In [17]:
analytics = analytics_service(initial_window=3)

snapshots = [
    next(analytics),
    analytics.send(10),
    analytics.send(20),
    analytics.send(30),
    analytics.send(40),
    analytics.send(("window", 2)),
    analytics.send(100),
    analytics.send("reset"),
    analytics.send(7),
]

final_report = capture_stop_value(lambda: analytics.send("stop"))

snapshots, final_report

([WindowSnapshot(values=(), window_size=3, mean=None),
  WindowSnapshot(values=(10.0,), window_size=3, mean=10.0),
  WindowSnapshot(values=(10.0, 20.0), window_size=3, mean=15.0),
  WindowSnapshot(values=(10.0, 20.0, 30.0), window_size=3, mean=20.0),
  WindowSnapshot(values=(20.0, 30.0, 40.0), window_size=3, mean=30.0),
  WindowSnapshot(values=(30.0, 40.0), window_size=2, mean=35.0),
  WindowSnapshot(values=(40.0, 100.0), window_size=2, mean=70.0),
  WindowSnapshot(values=(), window_size=2, mean=None),
  WindowSnapshot(values=(7.0,), window_size=2, mean=7.0)],
 AnalyticsReport(messages_processed=9, observations_seen=6, final_snapshot=WindowSnapshot(values=(7.0,), window_size=2, mean=7.0)))

In [18]:
assert snapshots[0].values == ()
assert snapshots[4].values == (20.0, 30.0, 40.0)
assert snapshots[5].values == (30.0, 40.0)
assert snapshots[6].values == (40.0, 100.0)
assert snapshots[7].values == ()
assert snapshots[8].mean == 7.0

assert final_report.observations_seen == 6
assert final_report.final_snapshot.values == (7.0,)

## Best-practice discussion

### Prime explicitly

A just-created generator cannot receive a non-`None` value:

```python
generator.send(10)
```

raises `TypeError`.

Prime it first with:

```python
next(generator)
```

### Use explicit message shapes

For larger protocols, prefer tagged tuples, dataclasses, or enums over ambiguous raw values.

### Keep completion separate

The final report is returned, not mixed into the live snapshot stream.

# Problem 4 — Forward exceptions with `throw` and recover selectively

## Scenario

A counter accepts integer step sizes through `send`.

The caller can also inject exceptions:

- `ValueError`: recover locally and continue;
- `KeyError`: treat it as a request to stop and return the current count;
- any other exception: let it propagate.

The delegator should remain transparent.

## Step 1 — Implement the resilient subgenerator

The exception is raised at the subgenerator's currently suspended `yield`.

That means a `try` statement must surround the `yield` that should receive injected exceptions.

In [19]:
def resilient_counter(start=0):
    count = start

    while True:
        try:
            step = yield count
        except ValueError as exc:
            yield f"recovered from ValueError: {exc}"
            continue
        except KeyError:
            return count

        if step is None:
            step = 1

        if not isinstance(step, int):
            raise TypeError("step must be an integer or None")

        count += step

In [20]:
def counter_service(start=0):
    final_count = yield from resilient_counter(start)
    return final_count

## Prediction checkpoint

Consider this sequence:

```python
g = counter_service(10)
next(g)
g.send(5)
g.throw(ValueError("temporary input failure"))
next(g)
g.send(2)
g.throw(KeyError("stop"))
```

Questions:

1. What value is yielded after `send(5)`?
2. What does the `ValueError` call return?
3. Why is one additional `next(g)` useful after recovery?
4. What value is attached to the final `StopIteration`?

In [21]:
counter = counter_service(10)

trace = []
trace.append(next(counter))
trace.append(counter.send(5))
trace.append(counter.throw(ValueError("temporary input failure")))
trace.append(next(counter))
trace.append(counter.send(2))

returned_count = capture_stop_value(
    lambda: counter.throw(KeyError("stop"))
)

trace, returned_count

([10, 15, 'recovered from ValueError: temporary input failure', 15, 17], 17)

In [22]:
assert trace == [
    10,
    15,
    "recovered from ValueError: temporary input failure",
    15,
    17,
]

assert returned_count == 17
assert getgeneratorstate(counter) == "GEN_CLOSED"

## Step 2 — Verify that unhandled exceptions escape

`yield from` does not suppress exceptions automatically.

If the subgenerator does not handle an injected exception, it propagates through the delegator.

In [23]:
counter = counter_service()

assert next(counter) == 0

try:
    counter.throw(RuntimeError("fatal"))
except RuntimeError as exc:
    escaped_message = str(exc)
else:
    raise AssertionError("RuntimeError should have propagated")

assert escaped_message == "fatal"
assert getgeneratorstate(counter) == "GEN_CLOSED"

escaped_message

'fatal'

## Design lesson

`throw` is useful when the caller needs to communicate an exceptional condition into the suspended computation.

Examples include:

- cancellation;
- invalid external data;
- rollback requests;
- timeout signals;
- recoverable transport failures.

Do not use exceptions as ordinary data when a regular command message would be clearer.

# Problem 5 — Trace `close()` through three delegation layers

## Scenario

A nested service has three generator layers:

1. outer service;
2. middle service;
3. leaf resource session.

Each layer records when it opens and closes.

Your task is to prove that closing the outer generator:

- propagates to the active subgenerator;
- executes every `finally` block;
- performs cleanup in last-in, first-out order.

## Step 1 — Implement the leaf resource

The leaf waits for commands and echoes them.

Its `finally` block represents mandatory resource cleanup.

In [24]:
def leaf_resource(events):
    events.append("leaf:open")

    try:
        while True:
            command = yield "leaf:ready"
            events.append(f"leaf:command:{command}")
    finally:
        events.append("leaf:close")

## Step 2 — Add two delegating layers

Each layer surrounds `yield from` with its own `try/finally`.

In [25]:
def middle_layer(events):
    events.append("middle:open")

    try:
        result = yield from leaf_resource(events)
        return result
    finally:
        events.append("middle:close")


def outer_layer(events):
    events.append("outer:open")

    try:
        result = yield from middle_layer(events)
        return result
    finally:
        events.append("outer:close")

## Prediction checkpoint

After this code:

```python
events = []
service = outer_layer(events)
next(service)
service.send("ping")
service.close()
```

Predict the exact event order.

Remember that `close()` injects `GeneratorExit` at the suspension point.

In [26]:
events = []
service = outer_layer(events)

assert next(service) == "leaf:ready"
assert service.send("ping") == "leaf:ready"

service.close()

events

['outer:open',
 'middle:open',
 'leaf:open',
 'leaf:command:ping',
 'leaf:close',
 'middle:close',
 'outer:close']

In [27]:
assert events == [
    "outer:open",
    "middle:open",
    "leaf:open",
    "leaf:command:ping",
    "leaf:close",
    "middle:close",
    "outer:close",
]

assert getgeneratorstate(service) == "GEN_CLOSED"

## Why cleanup order matters

Resources are usually acquired from outermost to innermost:

```text
outer -> middle -> leaf
```

They should be released in reverse order:

```text
leaf -> middle -> outer
```

This is the same structural idea used by:

- nested context managers;
- stack unwinding;
- transaction scopes;
- recursive resource acquisition.

## Critical rule: never yield after receiving `GeneratorExit`

A generator that catches `GeneratorExit` and yields another value violates the close protocol.

Python raises:

```text
RuntimeError: generator ignored GeneratorExit
```

The next debugging problem revisits this rule.

# Problem 6 — Compose protocol phases with multiple subgenerators

## Scenario

Build a secure key-value service with two phases:

1. authentication;
2. command processing.

The outer generator should delegate to an authentication subgenerator first. When authentication returns successfully, it should immediately delegate to the command processor.

This is a powerful use of generator return values: one subprotocol finishes and hands control to the next.

## Step 1 — Authentication subgenerator

The caller sends tokens.

The generator yields:

- `"TOKEN?"` initially;
- `"DENIED — TOKEN?"` after an incorrect token.

When the correct token arrives, the generator returns the number of attempts.

In [28]:
def authenticate(expected_token):
    attempts = 0
    prompt = "TOKEN?"

    while True:
        token = yield prompt
        attempts += 1

        if token == expected_token:
            return attempts

        prompt = "DENIED — TOKEN?"

## Step 2 — Command-processing subgenerator

Supported commands:

- `("set", key, value)`
- `("get", key)`
- `("delete", key)`
- `("quit",)`

The command processor yields a response after each non-terminal command.

On quit, it returns the final dictionary.

In [29]:
def key_value_commands():
    store = {}
    command = yield "COMMAND?"

    while True:
        if not isinstance(command, tuple) or not command:
            command = yield "ERROR: command must be a non-empty tuple"
            continue

        operation = command[0]

        if operation == "set" and len(command) == 3:
            _, key, value = command
            store[key] = value
            command = yield f"SET {key!r}"

        elif operation == "get" and len(command) == 2:
            _, key = command
            command = yield store.get(key, "<missing>")

        elif operation == "delete" and len(command) == 2:
            _, key = command
            existed = key in store
            store.pop(key, None)
            command = yield f"DELETED={existed}"

        elif operation == "quit" and len(command) == 1:
            return dict(store)

        else:
            command = yield f"ERROR: malformed {operation!r} command"

## Step 3 — Compose the phases

The secure service captures both subgenerator results:

- authentication attempts;
- final store.

It returns a combined session report.

In [30]:
@dataclass(frozen=True)
class SecureSessionReport:
    authentication_attempts: int
    final_store: dict


def secure_key_value_service(expected_token):
    attempts = yield from authenticate(expected_token)
    final_store = yield from key_value_commands()

    return SecureSessionReport(
        authentication_attempts=attempts,
        final_store=final_store,
    )

## Important protocol observation

When the correct token is sent, `authenticate` returns during that same `send(...)` call.

The outer generator then immediately enters `key_value_commands`, which yields `"COMMAND?"`.

Therefore:

```python
service.send(correct_token)
```

returns the first prompt of the **next phase**.

In [31]:
service = secure_key_value_service("s3cr3t")

conversation = [
    next(service),
    service.send("wrong"),
    service.send("still-wrong"),
    service.send("s3cr3t"),
    service.send(("set", "language", "Python")),
    service.send(("set", "level", "advanced")),
    service.send(("get", "language")),
    service.send(("delete", "level")),
]

session_report = capture_stop_value(
    lambda: service.send(("quit",))
)

conversation, session_report

(['TOKEN?',
  'DENIED — TOKEN?',
  'DENIED — TOKEN?',
  'COMMAND?',
  "SET 'language'",
  "SET 'level'",
  'Python',
  'DELETED=True'],
 SecureSessionReport(authentication_attempts=3, final_store={'language': 'Python'}))

In [32]:
assert conversation == [
    "TOKEN?",
    "DENIED — TOKEN?",
    "DENIED — TOKEN?",
    "COMMAND?",
    "SET 'language'",
    "SET 'level'",
    "Python",
    "DELETED=True",
]

assert session_report == SecureSessionReport(
    authentication_attempts=3,
    final_store={"language": "Python"},
)

## Design lesson: delegation is phase-oriented composition

A subgenerator can represent a complete protocol phase.

When it returns, the delegator can:

- inspect its result;
- choose the next subgenerator;
- repeat a phase;
- terminate;
- transform the result.

This technique is especially useful for interactive workflows and finite-state protocols.

# Problem 7 — Recursive delegation with aggregate return values

## Scenario

Traverse a nested structure containing dictionaries, lists, tuples, and scalar leaves.

For every leaf, yield:

```python
(path, value)
```

At the same time, each recursive generator should return aggregate statistics for its subtree:

- number of leaves;
- number of numeric leaves;
- sum of numeric leaves;
- maximum depth.

The parent combines the return values from its children.

## Step 1 — Define the aggregate type

We will keep the type immutable so accidental mutation cannot corrupt sibling results.

In [33]:
@dataclass(frozen=True)
class TreeStats:
    leaf_count: int
    numeric_count: int
    numeric_sum: float
    max_depth: int


EMPTY_TREE_STATS = TreeStats(
    leaf_count=0,
    numeric_count=0,
    numeric_sum=0.0,
    max_depth=0,
)

## Step 2 — Define a combination helper

The helper combines independent subtree summaries.

In [34]:
def combine_tree_stats(left, right):
    return TreeStats(
        leaf_count=left.leaf_count + right.leaf_count,
        numeric_count=left.numeric_count + right.numeric_count,
        numeric_sum=left.numeric_sum + right.numeric_sum,
        max_depth=max(left.max_depth, right.max_depth),
    )

## Step 3 — Implement the recursive generator

Rules:

- dictionary keys extend the path;
- list and tuple indices extend the path;
- every other object is treated as a leaf;
- booleans are not counted as numeric leaves, even though `bool` subclasses `int`.

In [35]:
def walk_nested(node, path=()):
    if isinstance(node, dict):
        combined = EMPTY_TREE_STATS

        for key, child in node.items():
            child_stats = yield from walk_nested(child, path + (key,))
            combined = combine_tree_stats(combined, child_stats)

        return combined

    if isinstance(node, (list, tuple)):
        combined = EMPTY_TREE_STATS

        for index, child in enumerate(node):
            child_stats = yield from walk_nested(child, path + (index,))
            combined = combine_tree_stats(combined, child_stats)

        return combined

    # Scalar leaf.
    yield path, node

    is_numeric = isinstance(node, (int, float)) and not isinstance(node, bool)

    return TreeStats(
        leaf_count=1,
        numeric_count=int(is_numeric),
        numeric_sum=float(node) if is_numeric else 0.0,
        max_depth=len(path),
    )

## Prediction checkpoint

For the structure below, determine:

- how many leaf records will be yielded;
- the sum of numeric leaves;
- the maximum path length.

Then run the traversal.

In [36]:
nested_data = {
    "user": {
        "name": "Ada",
        "scores": [10, 20, {"bonus": 5}],
    },
    "active": True,
    "coordinates": (1.5, -2.0),
}

records, tree_stats = collect_yields_and_return(
    walk_nested(nested_data)
)

records, tree_stats

([(('user', 'name'), 'Ada'),
  (('user', 'scores', 0), 10),
  (('user', 'scores', 1), 20),
  (('user', 'scores', 2, 'bonus'), 5),
  (('active',), True),
  (('coordinates', 0), 1.5),
  (('coordinates', 1), -2.0)],
 TreeStats(leaf_count=7, numeric_count=5, numeric_sum=34.5, max_depth=4))

In [37]:
assert records == [
    (("user", "name"), "Ada"),
    (("user", "scores", 0), 10),
    (("user", "scores", 1), 20),
    (("user", "scores", 2, "bonus"), 5),
    (("active",), True),
    (("coordinates", 0), 1.5),
    (("coordinates", 1), -2.0),
]

assert tree_stats == TreeStats(
    leaf_count=7,
    numeric_count=5,
    numeric_sum=34.5,
    max_depth=4,
)

## Why `yield from` is especially elegant here

Without `yield from`, the parent would need to:

1. manually iterate over the child's yielded records;
2. somehow capture the child's `StopIteration.value`;
3. distinguish ordinary exhaustion from other exceptions.

With `yield from`, both channels are handled directly:

```python
child_stats = yield from walk_nested(child, child_path)
```

The child's records flow outward, while its aggregate result flows inward.

# Problem 8 — Model an interactive workflow as generator states

## Scenario

Build a profile-creation workflow with three state subgenerators:

1. collect a non-empty name;
2. collect an integer age between 0 and 130;
3. ask for confirmation.

Each state:

- yields prompts;
- receives answers with `send`;
- returns validated data or a transition decision.

The top-level generator delegates to each state in sequence.

## Step 1 — Name state

The state repeats until it receives a non-empty string.

It returns the normalized name.

In [38]:
def collect_name():
    prompt = "Name?"

    while True:
        raw_name = yield prompt

        if isinstance(raw_name, str) and raw_name.strip():
            return raw_name.strip()

        prompt = "Name must be a non-empty string. Name?"

## Step 2 — Age state

The state accepts integers or strings representing integers.

It returns a valid age.

In [39]:
def collect_age():
    prompt = "Age?"

    while True:
        raw_age = yield prompt

        try:
            age = int(raw_age)
        except (TypeError, ValueError):
            prompt = "Age must be an integer. Age?"
            continue

        if 0 <= age <= 130:
            return age

        prompt = "Age must be between 0 and 130. Age?"

## Step 3 — Confirmation state

The state returns:

- `True` for yes;
- `False` for no.

It repeats on any other answer.

In [40]:
def confirm_profile(profile):
    prompt = f"Confirm {profile!r}? [yes/no]"

    while True:
        answer = yield prompt

        if isinstance(answer, str):
            normalized = answer.strip().lower()

            if normalized in {"yes", "y"}:
                return True

            if normalized in {"no", "n"}:
                return False

        prompt = "Please answer yes or no."

## Step 4 — Top-level workflow

If confirmation fails, restart from the name state.

If confirmation succeeds, return the completed profile.

In [41]:
def profile_workflow():
    while True:
        name = yield from collect_name()
        age = yield from collect_age()

        profile = {"name": name, "age": age}
        confirmed = yield from confirm_profile(profile)

        if confirmed:
            return profile

## Guided conversation

This example intentionally includes invalid inputs and one rejected profile.

In [42]:
workflow = profile_workflow()

dialogue = [
    next(workflow),
    workflow.send("   "),
    workflow.send("Grace"),
    workflow.send("not-an-age"),
    workflow.send("200"),
    workflow.send("36"),
    workflow.send("no"),
    workflow.send("Linus"),
    workflow.send(55),
]

completed_profile = capture_stop_value(
    lambda: workflow.send("yes")
)

dialogue, completed_profile

(['Name?',
  'Name must be a non-empty string. Name?',
  'Age?',
  'Age must be an integer. Age?',
  'Age must be between 0 and 130. Age?',
  "Confirm {'name': 'Grace', 'age': 36}? [yes/no]",
  'Name?',
  'Age?',
  "Confirm {'name': 'Linus', 'age': 55}? [yes/no]"],
 {'name': 'Linus', 'age': 55})

In [43]:
assert dialogue == [
    "Name?",
    "Name must be a non-empty string. Name?",
    "Age?",
    "Age must be an integer. Age?",
    "Age must be between 0 and 130. Age?",
    "Confirm {'name': 'Grace', 'age': 36}? [yes/no]",
    "Name?",
    "Age?",
    "Confirm {'name': 'Linus', 'age': 55}? [yes/no]",
]

assert completed_profile == {
    "name": "Linus",
    "age": 55,
}

## Architectural insight

Each subgenerator owns one cohesive concern:

- prompting;
- validation;
- retry behavior;
- return type.

The parent owns transition logic.

This separation makes a complex interactive protocol easier to test than one large generator containing every state and branch.

# Problem 9 — Manually reimplement the core `yield from` protocol

## Scenario

To understand delegation deeply, implement an educational replacement for:

```python
result = yield from subgenerator
```

The manual implementation must handle:

- initial priming;
- `next`;
- `send`;
- `throw`;
- `close`;
- `StopIteration.value`.

This is an advanced exercise based on the semantics formalized by PEP 380.

> Production code should use the real `yield from`.  
> The manual version is for learning and testing your mental model.

## Step 1 — Read the protocol as a state machine

The delegator stores the most recently yielded subgenerator value in `current`.

Then it waits for one of four caller actions:

1. `next()` / `send(None)`;
2. `send(non_none_value)`;
3. `throw(exception)`;
4. `close()`.

The delegator maps each action to the corresponding subiterator method.

In [44]:
def manual_yield_from(iterable):
    """Educational implementation of the essential yield-from protocol."""
    iterator = iter(iterable)

    try:
        current = next(iterator)
    except StopIteration as stop:
        return stop.value

    while True:
        try:
            sent = yield current

        except GeneratorExit:
            close_method = getattr(iterator, "close", None)

            if close_method is not None:
                close_method()

            raise

        except BaseException as exc:
            throw_method = getattr(iterator, "throw", None)

            if throw_method is None:
                raise

            try:
                current = throw_method(
                    type(exc),
                    exc,
                    exc.__traceback__,
                )
            except StopIteration as stop:
                return stop.value

        else:
            try:
                if sent is None:
                    current = next(iterator)
                else:
                    # This intentionally raises AttributeError if the
                    # delegated iterator does not implement send.
                    current = iterator.send(sent)
            except StopIteration as stop:
                return stop.value

## Step 2 — Create a protocol probe

The worker records exceptional events so the two implementations can be compared.

Protocol:

- first yield: `"ready"`;
- ordinary messages: yield `"echo:<message>"`;
- injected `LookupError`: yield `"recovered"`;
- `"stop"`: return `"worker-result"`;
- close: record cleanup.

In [45]:
def protocol_worker(log):
    try:
        message = yield "ready"

        while True:
            try:
                if message == "stop":
                    return "worker-result"

                message = yield f"echo:{message}"

            except LookupError as exc:
                log.append(f"lookup:{exc}")
                message = yield "recovered"

    finally:
        log.append("worker:finally")

In [46]:
def native_wrapper(log):
    result = yield from protocol_worker(log)
    return result


def manual_wrapper(log):
    result = yield from manual_yield_from(protocol_worker(log))
    return result

## Step 3 — Drive both implementations with the same interaction

The transcript function exercises:

- priming;
- `send`;
- `throw`;
- normal return.

In [47]:
def run_protocol_transcript(wrapper):
    log = []
    generator = wrapper(log)

    transcript = [
        next(generator),
        generator.send("alpha"),
        generator.throw(LookupError("temporary")),
        generator.send("beta"),
    ]

    result = capture_stop_value(
        lambda: generator.send("stop")
    )

    return transcript, result, log


native_transcript = run_protocol_transcript(native_wrapper)
manual_transcript = run_protocol_transcript(manual_wrapper)

native_transcript, manual_transcript

C:\Users\user1\AppData\Local\Temp\ipykernel_19396\2518878616.py:29: DeprecationWarning: the (type, exc, tb) signature of throw() is deprecated, use the single-arg signature instead.
  current = throw_method(


((['ready', 'echo:alpha', 'recovered', 'echo:beta'],
  'worker-result',
  ['lookup:temporary', 'worker:finally']),
 (['ready', 'echo:alpha', 'recovered', 'echo:beta'],
  'worker-result',
  ['lookup:temporary', 'worker:finally']))

In [48]:
assert native_transcript == manual_transcript

assert native_transcript == (
    [
        "ready",
        "echo:alpha",
        "recovered",
        "echo:beta",
    ],
    "worker-result",
    [
        "lookup:temporary",
        "worker:finally",
    ],
)

## Step 4 — Compare close propagation

Both wrappers must close the worker and execute its `finally` block.

In [49]:
def close_transcript(wrapper):
    log = []
    generator = wrapper(log)

    assert next(generator) == "ready"
    generator.close()

    return log, getgeneratorstate(generator)


native_close = close_transcript(native_wrapper)
manual_close = close_transcript(manual_wrapper)

assert native_close == manual_close
assert native_close == (["worker:finally"], "GEN_CLOSED")

native_close

(['worker:finally'], 'GEN_CLOSED')

## What this exercise reveals

`yield from` is equivalent to a carefully implemented protocol loop—not a simple iteration loop.

It coordinates:

- values;
- sent data;
- injected exceptions;
- finalization;
- completion results.

This is why replacing `yield from` with a normal `for` loop silently changes semantics.

# Problem 10 — Debugging laboratory: lifecycle and protocol failures

This problem contains several intentionally incorrect assumptions.

For each case:

1. predict the exception or state;
2. run the demonstration;
3. read the explanation;
4. apply the safer pattern.

## Case A — Sending a value to a just-created generator

A generator begins in the `GEN_CREATED` state.

It has not yet reached a `yield`, so there is no suspended expression waiting to receive a non-`None` value.

In [50]:
def inbox():
    message = yield "ready"
    yield f"received:{message}"


created = inbox()
state_before = getgeneratorstate(created)

try:
    created.send("hello")
except TypeError as exc:
    case_a_error = str(exc)
else:
    raise AssertionError("Expected TypeError")
finally:
    created.close()

state_before, case_a_error

('GEN_CREATED', "can't send non-None value to a just-started generator")

### Solution

Prime first:

```python
g = inbox()
next(g)
g.send("hello")
```

`next(g)` is equivalent to `g.send(None)`, which is allowed in the created state.

In [51]:
primed = inbox()

assert next(primed) == "ready"
assert primed.send("hello") == "received:hello"

primed.close()

## Case B — Sending through `yield from` to a plain list iterator

A list iterator supports `__next__`, but it does not support `.send(...)`.

Ordinary iteration works. Non-`None` sends do not.

In [52]:
def delegate_to_list():
    yield from [10, 20, 30]


list_delegate = delegate_to_list()

assert next(list_delegate) == 10

try:
    list_delegate.send("unexpected")
except AttributeError as exc:
    case_b_error = str(exc)
else:
    raise AssertionError("Expected AttributeError")
finally:
    list_delegate.close()

case_b_error

"'list_iterator' object has no attribute 'send'"

### Solution

Only use non-`None` sends when the active delegated object implements the generator protocol.

A plain iterable is suitable for one-way value forwarding, not two-way coroutine communication.

## Case C — Closing a generator before it starts

If a generator is closed while still in `GEN_CREATED`, its body never begins.

That means setup code inside the body does not run—and neither does its `finally` block, because the `try` statement was never entered.

In [53]:
def never_started(events):
    events.append("body:started")

    try:
        yield "running"
    finally:
        events.append("body:cleanup")


events = []
g = never_started(events)

assert getgeneratorstate(g) == "GEN_CREATED"

g.close()

assert events == []
assert getgeneratorstate(g) == "GEN_CLOSED"

events

[]

### Design implication

Do not assume that a generator's `finally` block will run unless execution actually entered its protected region.

For critical resource acquisition, prefer a context manager when the lifecycle is naturally block-scoped.

## Case D — Yielding after `GeneratorExit`

This generator incorrectly catches `GeneratorExit` and yields another value.

In [54]:
def broken_cleanup():
    try:
        yield "ready"
    except GeneratorExit:
        yield "illegal"


broken = broken_cleanup()
assert next(broken) == "ready"

try:
    broken.close()
except RuntimeError as exc:
    case_d_error = str(exc)
else:
    raise AssertionError("Expected RuntimeError")

# The failed close left the generator suspended at the illegal yield.
# Advance it once so its frame can finish cleanly for the notebook.
try:
    next(broken)
except StopIteration:
    pass

case_d_error

'generator ignored GeneratorExit'

### Solution

Perform cleanup and re-raise implicitly by allowing `GeneratorExit` to continue:

```python
def correct_cleanup():
    try:
        yield "ready"
    finally:
        release_resource()
```

A `finally` block is usually the clearest solution.

## Case E — Confusing a yielded value with a returned value

A generator's `return value` does not appear in a normal `for` loop.

In [55]:
def yields_then_returns():
    yield "data"
    return "completion"


loop_items = list(yields_then_returns())
all_items, completion = collect_yields_and_return(yields_then_returns())

assert loop_items == ["data"]
assert all_items == ["data"]
assert completion == "completion"

loop_items, completion

(['data'], 'completion')

### Solution

Use `yield from` inside another generator—or explicitly catch `StopIteration`—when the return value matters.

# Problem 11 — Capstone: transactional editing with commit, rollback, and cleanup

## Scenario

Build an in-memory transaction protocol.

The transaction starts from an original dictionary and accepts commands:

- `("set", key, value)`
- `("delete", key)`
- `("read", key)`
- `("commit",)`

The caller may also inject `Rollback(reason)` with `throw`.

Requirements:

- edits apply to a working copy;
- commit returns the working copy;
- rollback returns the untouched original dictionary;
- a transaction log records begin, changes, commit/rollback, and end;
- a service delegator captures the result and returns a structured report.

## Step 1 — Define the rollback signal and report type

In [56]:
class Rollback(Exception):
    """Signal that the active transaction should be rolled back."""


@dataclass(frozen=True)
class TransactionReport:
    status: str
    data: dict
    log: tuple[str, ...]

## Step 2 — Implement the transaction subgenerator

The active transaction catches `Rollback`.

Any unrelated exception remains fatal and propagates.

In [57]:
def transaction(original, log):
    original_copy = dict(original)
    working = dict(original)

    log.append("BEGIN")

    command = yield dict(working)

    try:
        while True:
            if not isinstance(command, tuple) or not command:
                command = yield "ERROR: malformed command"
                continue

            operation = command[0]

            if operation == "set" and len(command) == 3:
                _, key, value = command
                working[key] = value
                log.append(f"SET {key!r}")
                command = yield dict(working)

            elif operation == "delete" and len(command) == 2:
                _, key = command
                working.pop(key, None)
                log.append(f"DELETE {key!r}")
                command = yield dict(working)

            elif operation == "read" and len(command) == 2:
                _, key = command
                command = yield working.get(key, "<missing>")

            elif operation == "commit" and len(command) == 1:
                log.append("COMMIT")
                return "committed", dict(working)

            else:
                command = yield f"ERROR: unsupported command {command!r}"

    except Rollback as exc:
        log.append(f"ROLLBACK: {exc}")
        return "rolled_back", original_copy

    finally:
        log.append("END")

## Step 3 — Add the service delegator

The service owns the log and converts the subgenerator's tuple result into a dataclass.

In [58]:
def transaction_service(original):
    log = []

    status, data = yield from transaction(original, log)

    return TransactionReport(
        status=status,
        data=data,
        log=tuple(log),
    )

## Step 4 — Commit path

Follow a normal edit-and-commit session.

In [59]:
commit_session = transaction_service({"mode": "safe", "retries": 2})

commit_trace = [
    next(commit_session),
    commit_session.send(("set", "retries", 5)),
    commit_session.send(("set", "timeout", 30)),
    commit_session.send(("delete", "mode")),
    commit_session.send(("read", "timeout")),
]

commit_report = capture_stop_value(
    lambda: commit_session.send(("commit",))
)

commit_trace, commit_report

([{'mode': 'safe', 'retries': 2},
  {'mode': 'safe', 'retries': 5},
  {'mode': 'safe', 'retries': 5, 'timeout': 30},
  {'retries': 5, 'timeout': 30},
  30],
 TransactionReport(status='committed', data={'retries': 5, 'timeout': 30}, log=('BEGIN', "SET 'retries'", "SET 'timeout'", "DELETE 'mode'", 'COMMIT', 'END')))

In [60]:
assert commit_report == TransactionReport(
    status="committed",
    data={"retries": 5, "timeout": 30},
    log=(
        "BEGIN",
        "SET 'retries'",
        "SET 'timeout'",
        "DELETE 'mode'",
        "COMMIT",
        "END",
    ),
)

## Step 5 — Rollback path

Inject `Rollback` into the delegator.

Because `yield from` forwards `throw`, the transaction subgenerator catches it.

In [61]:
rollback_session = transaction_service({"enabled": True, "limit": 10})

assert next(rollback_session) == {"enabled": True, "limit": 10}
assert rollback_session.send(("set", "limit", 999)) == {
    "enabled": True,
    "limit": 999,
}
assert rollback_session.send(("delete", "enabled")) == {
    "limit": 999,
}

rollback_report = capture_stop_value(
    lambda: rollback_session.throw(
        Rollback("validation failed")
    )
)

rollback_report

TransactionReport(status='rolled_back', data={'enabled': True, 'limit': 10}, log=('BEGIN', "SET 'limit'", "DELETE 'enabled'", 'ROLLBACK: validation failed', 'END'))

In [62]:
assert rollback_report == TransactionReport(
    status="rolled_back",
    data={"enabled": True, "limit": 10},
    log=(
        "BEGIN",
        "SET 'limit'",
        "DELETE 'enabled'",
        "ROLLBACK: validation failed",
        "END",
    ),
)

## Capstone discussion

This single example combines nearly every major `yield from` feature:

- yielded working-state snapshots flow outward;
- commands sent by the caller flow inward;
- a rollback exception is forwarded inward;
- a structured completion value flows back to the delegator;
- `finally` guarantees end-of-transaction logging;
- the outer service converts the subgenerator result into a domain-specific report.

# Additional mini-problems with solutions

These smaller exercises reinforce subtle details without introducing a large new scenario.

## Mini-problem A — Capture the return value of an empty subgenerator

What happens when a subgenerator returns before yielding anything?

In [63]:
def empty_worker():
    if False:
        yield None
    return 42


def empty_delegator():
    value = yield from empty_worker()
    yield f"captured:{value}"
    return value


yielded, returned = collect_yields_and_return(empty_delegator())

assert yielded == ["captured:42"]
assert returned == 42

yielded, returned

(['captured:42'], 42)

### Explanation

`yield from` handles immediate `StopIteration` during initial priming and extracts its `.value`.

## Mini-problem B — Delegate to a string

Strings are iterables, so `yield from` can flatten them character by character.

In [64]:
def spell(words):
    for word in words:
        yield from word
        yield " "


assert "".join(spell(["yield", "from"])) == "yield from "

list(spell(["go"]))

['g', 'o', ' ']

### Explanation

Delegation does not require a generator. It can target any iterable for one-way iteration.

However, a non-generator iterator usually cannot receive non-`None` values through `send`.

## Mini-problem C — Inspect the active subgenerator

A delegating generator exposes its current delegation target through `gi_yieldfrom`.

In [65]:
def inner_probe():
    yield "inner-value"


def outer_probe():
    yield from inner_probe()


outer = outer_probe()

assert outer.gi_yieldfrom is None
assert next(outer) == "inner-value"

active_subgenerator = outer.gi_yieldfrom

assert active_subgenerator is not None
assert getgeneratorstate(active_subgenerator) == "GEN_SUSPENDED"

try:
    next(outer)
except StopIteration:
    pass

assert outer.gi_yieldfrom is None

type(active_subgenerator).__name__

'generator'

### Explanation

`gi_yieldfrom` is useful for debugging and introspection.

Avoid depending on it for ordinary application control flow; it is an implementation-facing inspection attribute.

## Mini-problem D — A delegator can continue after a subgenerator returns

The subgenerator's completion closes only the subgenerator. The delegator continues from the statement after `yield from`.

In [66]:
def two_item_worker():
    yield "A"
    yield "B"
    return "worker-done"


def continuing_delegator():
    result = yield from two_item_worker()
    yield f"after:{result}"
    return "delegator-done"


yielded, returned = collect_yields_and_return(
    continuing_delegator()
)

assert yielded == ["A", "B", "after:worker-done"]
assert returned == "delegator-done"

yielded, returned

(['A', 'B', 'after:worker-done'], 'delegator-done')

## Mini-problem E — Repeatedly delegate to fresh subgenerators

A completed generator cannot be restarted, but a delegator can create a new subgenerator for each round.

In [67]:
def finite_round(round_number):
    yield f"round-{round_number}:start"
    yield f"round-{round_number}:end"
    return round_number * 10


def tournament(round_count):
    scores = []

    for round_number in range(1, round_count + 1):
        score = yield from finite_round(round_number)
        scores.append(score)

    return scores


yielded, scores = collect_yields_and_return(tournament(3))

assert yielded == [
    "round-1:start",
    "round-1:end",
    "round-2:start",
    "round-2:end",
    "round-3:start",
    "round-3:end",
]

assert scores == [10, 20, 30]

yielded, scores

(['round-1:start',
  'round-1:end',
  'round-2:start',
  'round-2:end',
  'round-3:start',
  'round-3:end'],
 [10, 20, 30])

# Consolidated best practices

## 1. Treat a generator as a protocol, not merely an iterator

Document:

- what it yields;
- what it accepts through `send`;
- which exceptions it handles through `throw`;
- what it returns;
- what cleanup occurs on `close`.

## 2. Prime before sending non-`None`

```python
g = protocol()
initial_output = next(g)
response = g.send(message)
```

## 3. Use `yield from` for transparent delegation

It forwards the complete generator protocol.

A normal loop only forwards iterated values.

## 4. Use return values for completion metadata

Reserve yielded values for the stream.

Return summaries, statuses, counts, and final results.

## 5. Put cleanup in `finally`

This handles:

- normal completion;
- injected exceptions;
- explicit close;
- failures in nested generators.

## 6. Do not yield after `GeneratorExit`

Cleanup and allow termination to continue.

## 7. Keep subgenerators cohesive

Good subgenerators often represent:

- one protocol phase;
- one parser stage;
- one recursive subtree;
- one transaction scope;
- one state in a workflow.

## 8. Avoid overusing generator coroutines

For modern asynchronous I/O, use `async def`, `await`, and asynchronous generators.

Generator-based protocols remain valuable for:

- synchronous streaming;
- recursive traversal;
- educational state machines;
- lazy transformation;
- compact, composable control flow.

# Final mental model

When execution is suspended inside:

```python
result = yield from subgenerator
```

imagine a temporary direct channel:

```text
caller  <============================>  subgenerator
             delegator forwards
```

The delegator becomes active again when:

- the subgenerator returns;
- an unhandled exception escapes;
- the delegator is closed.

The returned value then appears as:

```python
result
```

That single mental model explains the behavior of `next`, `send`, `throw`, `close`, yielded values, and `StopIteration.value`.

# Suggested extension exercise

Design a text-adventure engine where each room is a subgenerator.

Each room should:

- yield a description;
- accept commands through `send`;
- return the identifier of the next room;
- raise or catch a custom exception for invalid movement;
- clean up room-specific resources in `finally`.

The world controller should repeatedly select the next room with `yield from` until the player reaches an exit state.